# 이번 AASIST 학습 목표

- 로컬 음원을 Colab에서 정상적으로 읽는가
- 모든 음원이 [64600] 크기로 들어가는가
- T4에서 AASIST가 OOM 없이 학습되는가
- Validation metric이 정상적으로 계산되는가

# 필요 라이브러리 install

In [1]:
# GPU 확인

!nvidia-smi

import sys
import torch
import platform

print("Python :", sys.version)
print("OS     :", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.version.cuda)
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Thu Aug 20 03:37:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 시스템 패키지 설치

# Codec augmentation을 위해 FFmpeg를 설치합니다.

!apt-get update -qq
!apt-get install -y -qq ffmpeg libsndfile1

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [3]:
%pip install -q -U \
    numpy \
    pandas \
    scipy \
    scikit-learn \
    librosa \
    soundfile \
    soxr \
    matplotlib \
    tqdm \
    pyyaml \
    joblib \
    einops \
    tensorboard

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9

In [4]:
# XLS-R용 Hugging Face 설치
%pip install -q -U \
    transformers \
    accelerate \
    datasets \
    huggingface_hub \
    safetensors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.2/793.2 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 17.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.67.0 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.67.0 which is incompatible.


In [5]:
XLSR_MODEL_NAME = "facebook/wav2vec2-xls-r-300m"

In [6]:
# AASIST 설치

!git clone -q https://github.com/clovaai/aasist.git /content/aasist

In [7]:
# RawBoost 설치

!git clone -q \
    https://github.com/TakHemlata/RawBoost-antispoofing.git \
    /content/RawBoost-antispoofing

In [ ]:
# mamba 설치
%pip install -q -U ninja packaging
%pip install -q "mamba-ssm[causal-conv1d]" --no-build-isolation

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.4/216.4 kB 6.9 MB/s eta 0:00:00
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.5/728.5 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 MB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 772.4/772.4 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.3/29.3 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.8/897.8 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# 전체 Import

# ============================================================
# 0. Python Standard Library
# ============================================================

import os
import sys
import gc
import math
import json
import time
import copy
import random
import shutil
import tempfile
import subprocess

from pathlib import Path
from typing import (
    Dict,
    List,
    Tuple,
    Optional,
    Union,
    Any,
)
from dataclasses import dataclass


# ============================================================
# 1. Numeric / Data
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 2. Signal Processing
# ============================================================

import scipy
from scipy import signal
from scipy.optimize import minimize, brentq


# ============================================================
# 3. Audio
# ============================================================

import librosa
import soundfile as sf
import torchaudio

from torchaudio.transforms import (
    Resample,
    MelSpectrogram,
    AmplitudeToDB,
)


# ============================================================
# 4. PyTorch
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch import Tensor

from torch.utils.data import (
    Dataset,
    DataLoader,
    WeightedRandomSampler,
)

from torch.optim import (
    Adam,
    AdamW,
)

from torch.optim.lr_scheduler import (
    CosineAnnealingLR,
    ReduceLROnPlateau,
)


# ============================================================
# 5. AMP
# ============================================================

from torch.amp import autocast, GradScaler


# ============================================================
# 6. Hugging Face / XLS-R
# ============================================================

from transformers import (
    AutoModel,
    AutoProcessor,
    AutoFeatureExtractor,
    Wav2Vec2Model,
    Wav2Vec2FeatureExtractor,
    get_cosine_schedule_with_warmup,
)

from accelerate import Accelerator


# ============================================================
# 7. sklearn
# ============================================================

from sklearn.model_selection import (
    StratifiedKFold,
    GroupKFold,
    StratifiedGroupKFold,
    train_test_split,
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    confusion_matrix,
    classification_report,
    log_loss,
)


# ============================================================
# 8. Utilities
# ============================================================

import yaml
import joblib

from tqdm.auto import tqdm

from einops import (
    rearrange,
    repeat,
)

import matplotlib.pyplot as plt


# ============================================================
# 9. Mamba
# ============================================================

try:
    from mamba_ssm import Mamba
    MAMBA_AVAILABLE = True
except ImportError:
    MAMBA_AVAILABLE = False
    print("Mamba is not installed.")


# ============================================================
# 10. Device
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device :", DEVICE)

if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))
    print(
        "VRAM   :",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3,
            2
        ),
        "GB"
    )

In [ ]:
# AASIST IMPORT

!rm -rf /content/aasist

!git clone -q \
    https://github.com/clovaai/aasist.git \
    /content/aasist

print("AASIST repository clone 완료")

In [ ]:
# RandomSeed 설정
SEED = 42


def seed_everything(seed=42):

    random.seed(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = True


seed_everything(SEED)

print("Seed :", SEED)

In [ ]:
# datasets zip 파일 업로드 및 압축해제

from google.colab import files

uploaded = files.upload()

In [ ]:
print(uploaded.keys())

In [ ]:
# 압축해제

UPLOAD_DIR = Path("/content")
DATA_DIR = Path("/content/data")

DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)


zip_files = list(
    UPLOAD_DIR.glob("*.zip")
)


print("ZIP files :", zip_files)


for zip_path in zip_files:

    print("압축 해제 :", zip_path)

    with zipfile.ZipFile(
        zip_path,
        "r"
    ) as zip_ref:

        zip_ref.extractall(DATA_DIR)


print("압축 해제 완료")

In [ ]:
# 폴더확인

!find /content/data -maxdepth 3 -type f | head -30

In [ ]:
# train.csv 확인
csv_candidates = list(
    DATA_DIR.rglob("train.csv")
)

print(csv_candidates)

assert len(csv_candidates) > 0, \
    "train.csv를 찾을 수 없습니다."

CSV_PATH = csv_candidates[0]

print("CSV_PATH :", CSV_PATH)

In [ ]:
df = pd.read_csv(CSV_PATH)

print("shape :", df.shape)

display(df.head())

In [ ]:
# 데이터설정
# ============================================================
# 반드시 데이터에 맞게 확인
# ============================================================

FILE_COL = "path"

LABEL_COL = "label"


# ============================================================
# Audio
# ============================================================

SAMPLE_RATE = 16000

# AASIST 공식 설정
MAX_LEN = 64600


# ============================================================
# Training
# ============================================================

BATCH_SIZE = 8

NUM_WORKERS = 2

EPOCHS = 10

LEARNING_RATE = 1e-4

WEIGHT_DECAY = 1e-4


# ============================================================
# Quick baseline
# ============================================================

QUICK_TEST = True

MAX_SAMPLES = 25000


# Peak Normalize
# 우선 공식 AASIST에 가깝게 False
NORMALIZE = False


print(
    f"Audio duration = "
    f"{MAX_LEN / SAMPLE_RATE:.3f} sec"
)

In [ ]:
REAL_LABELS = {
    "real",
    "bonafide",
    "bona-fide",
    "genuine",
    "human"
}

FAKE_LABELS = {
    "fake",
    "spoof",
    "deepfake",
    "synthetic",
    "generated"
}


def convert_label(x):

    # 이미 0 / 1이면 그대로 사용
    if isinstance(
        x,
        (
            int,
            np.integer,
            float,
            np.floating
        )
    ):
        return int(x)

    x = str(x).strip().lower()

    if x in REAL_LABELS:
        return 0

    if x in FAKE_LABELS:
        return 1

    raise ValueError(
        f"알 수 없는 label: {x}"
    )


df["_label"] = df[LABEL_COL].apply(
    convert_label
)


print(
    df["_label"].value_counts()
)

print(
    df["_label"].value_counts(
        normalize=True
    )
)

In [ ]:
# audio path 만들기
BASE_DIR = CSV_PATH.parent


def resolve_audio_path(p):

    p = Path(str(p))

    candidates = [
        p,
        BASE_DIR / p,
        BASE_DIR / "audio" / p,
        DATA_DIR / p,
    ]

    for candidate in candidates:

        if candidate.exists():
            return str(candidate)

    return None


df["_audio_path"] = df[FILE_COL].apply(
    resolve_audio_path
)


missing = df["_audio_path"].isna().sum()

print("전체 :", len(df))
print("찾지 못한 Audio :", missing)

In [ ]:
if (
    QUICK_TEST
    and len(df) > MAX_SAMPLES
):

    df, _ = train_test_split(
        df,
        train_size=MAX_SAMPLES,
        stratify=df["_label"],
        random_state=SEED,
    )

    df = df.reset_index(
        drop=True
    )


print("사용할 데이터 :", len(df))

print(
    df["_label"].value_counts()
)

In [ ]:
# train / validation split
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["_label"],
    random_state=SEED,
)


train_df = train_df.reset_index(
    drop=True
)

val_df = val_df.reset_index(
    drop=True
)


print(
    "Train :",
    len(train_df)
)

print(
    "Valid :",
    len(val_df)
)

print()

print(
    "Train label"
)

print(
    train_df["_label"].value_counts()
)

print()

print(
    "Valid label"
)

print(
    val_df["_label"].value_counts()
)

In [ ]:
# audio 전처리
def pad_or_crop(
    x,
    max_len=64600,
    train=True
):

    x = np.asarray(
        x,
        dtype=np.float32
    )

    x_len = len(x)

    # 빈 audio 방어
    if x_len == 0:
        return np.zeros(
            max_len,
            dtype=np.float32
        )

    # ========================================================
    # Audio가 길 경우
    # ========================================================

    if x_len >= max_len:

        if train:

            start = np.random.randint(
                0,
                x_len - max_len + 1
            )

        else:

            # validation은 deterministic
            start = 0

        return x[
            start:start + max_len
        ]

    # ========================================================
    # Audio가 짧을 경우 반복 padding
    # ========================================================

    repeat = (
        max_len // x_len
    ) + 1

    x = np.tile(
        x,
        repeat
    )

    return x[:max_len]

In [ ]:
# audiodata class
class DeepVoiceDataset(Dataset):

    def __init__(
        self,
        dataframe,
        sample_rate=16000,
        max_len=64600,
        train=True,
        normalize=False,
    ):

        self.df = dataframe.reset_index(
            drop=True
        )

        self.sample_rate = sample_rate

        self.max_len = max_len

        self.train = train

        self.normalize = normalize


    def __len__(self):

        return len(self.df)


    def __getitem__(
        self,
        index
    ):

        row = self.df.iloc[index]

        path = row["_audio_path"]

        label = int(
            row["_label"]
        )


        # ====================================================
        # Audio load
        # ====================================================

        audio, sr = sf.read(
            path,
            dtype="float32",
            always_2d=False,
        )


        # ====================================================
        # Stereo → Mono
        # ====================================================

        if audio.ndim > 1:

            audio = audio.mean(
                axis=1
            )


        # ====================================================
        # Resampling
        # ====================================================

        if sr != self.sample_rate:

            audio = librosa.resample(
                audio,
                orig_sr=sr,
                target_sr=self.sample_rate,
            )


        audio = np.asarray(
            audio,
            dtype=np.float32
        )


        # ====================================================
        # Optional normalization
        # ====================================================

        if self.normalize:

            peak = np.max(
                np.abs(audio)
            )

            if peak > 0:

                audio = (
                    audio
                    /
                    (peak + 1e-8)
                )


        # ====================================================
        # Crop / Padding
        # ====================================================

        audio = pad_or_crop(
            audio,
            max_len=self.max_len,
            train=self.train,
        )


        audio = torch.tensor(
            audio,
            dtype=torch.float32
        )


        return (
            audio,
            torch.tensor(
                label,
                dtype=torch.long
            )
        )

In [ ]:
# dataloader
train_dataset = DeepVoiceDataset(
    train_df,
    sample_rate=SAMPLE_RATE,
    max_len=MAX_LEN,
    train=True,
    normalize=NORMALIZE,
)


val_dataset = DeepVoiceDataset(
    val_df,
    sample_rate=SAMPLE_RATE,
    max_len=MAX_LEN,
    train=False,
    normalize=NORMALIZE,
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True,
    persistent_workers=(
        NUM_WORKERS > 0
    ),
)


val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
    persistent_workers=(
        NUM_WORKERS > 0
    ),
)


print(
    "Train batches :",
    len(train_loader)
)

print(
    "Valid batches :",
    len(val_loader)
)

In [ ]:
# dataloader 동작확인

batch_x, batch_y = next(
    iter(train_loader)
)


print(
    "Audio batch :",
    batch_x.shape
)

print(
    "Label batch :",
    batch_y.shape
)

print(
    "Labels :",
    batch_y
)

print(
    "Min :",
    batch_x.min().item()
)

print(
    "Max :",
    batch_x.max().item()
)

In [ ]:
# aasist import

AASIST_ROOT = "/content/aasist"

if AASIST_ROOT not in sys.path:
    sys.path.insert(
        0,
        AASIST_ROOT
    )


from models.AASIST import Model as AASIST

In [ ]:
# AASIST Config 읽기

CONFIG_PATH = (
    "/content/aasist/config/AASIST.conf"
)


with open(
    CONFIG_PATH,
    "r"
) as f:

    aasist_config = json.load(f)


model_config = (
    aasist_config[
        "model_config"
    ]
)


print(
    json.dumps(
        model_config,
        indent=2
    )
)

In [ ]:
# AASIST 생성
model = AASIST(
    model_config
)

model = model.to(
    DEVICE
)


num_params = sum(
    p.numel()
    for p in model.parameters()
)


trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print(
    f"Total params : "
    f"{num_params:,}"
)

print(
    f"Trainable    : "
    f"{trainable_params:,}"
)

In [ ]:
# 학습전 ForwardTest

model.eval()


batch_x, batch_y = next(
    iter(train_loader)
)


batch_x = batch_x[:2].to(
    DEVICE
)


with torch.no_grad():

    hidden, logits = model(
        batch_x
    )


print(
    "Input  :",
    batch_x.shape
)

print(
    "Hidden :",
    hidden.shape
)

print(
    "Logits :",
    logits.shape
)

In [ ]:
# loss + Optimizer
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

In [ ]:
USE_AMP = True


scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(
        USE_AMP
        and DEVICE.type == "cuda"
    ),
)

In [ ]:
# EER 계산함수
def calculate_eer(
    y_true,
    y_score
):

    fpr, tpr, thresholds = roc_curve(
        y_true,
        y_score,
        pos_label=1,
    )

    fnr = 1 - tpr

    index = np.nanargmin(
        np.abs(
            fnr - fpr
        )
    )

    eer = (
        fpr[index]
        +
        fnr[index]
    ) / 2

    threshold = thresholds[
        index
    ]

    return eer, threshold

In [ ]:
# Train 함수
def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    scaler,
    device,
):

    model.train()

    running_loss = 0.0

    preds = []
    targets = []


    progress = tqdm(
        loader,
        desc="Train",
        leave=False
    )


    for audio, label in progress:

        audio = audio.to(
            device,
            non_blocking=True
        )

        label = label.to(
            device,
            non_blocking=True
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(
                USE_AMP
                and device.type == "cuda"
            ),
        ):

            _, logits = model(
                audio,
                Freq_aug=False
            )

            loss = criterion(
                logits,
                label
            )


        scaler.scale(
            loss
        ).backward()


        scaler.step(
            optimizer
        )


        scaler.update()


        running_loss += (
            loss.item()
            *
            audio.size(0)
        )


        pred = logits.argmax(
            dim=1
        )


        preds.extend(
            pred.detach()
            .cpu()
            .numpy()
        )

        targets.extend(
            label.detach()
            .cpu()
            .numpy()
        )


        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )


    epoch_loss = (
        running_loss
        /
        len(loader.dataset)
    )


    accuracy = accuracy_score(
        targets,
        preds
    )


    return (
        epoch_loss,
        accuracy
    )

In [ ]:
# validation 함수
def validate(
    model,
    loader,
    criterion,
    device,
):

    model.eval()

    running_loss = 0.0

    all_labels = []
    all_preds = []
    all_probs = []


    with torch.no_grad():

        progress = tqdm(
            loader,
            desc="Valid",
            leave=False
        )


        for audio, label in progress:

            audio = audio.to(
                device,
                non_blocking=True
            )

            label = label.to(
                device,
                non_blocking=True
            )


            with torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(
                    USE_AMP
                    and device.type == "cuda"
                ),
            ):

                _, logits = model(
                    audio
                )

                loss = criterion(
                    logits,
                    label
                )


            prob = torch.softmax(
                logits.float(),
                dim=1
            )[:, 1]


            pred = logits.argmax(
                dim=1
            )


            running_loss += (
                loss.item()
                *
                audio.size(0)
            )


            all_labels.extend(
                label.cpu().numpy()
            )

            all_preds.extend(
                pred.cpu().numpy()
            )

            all_probs.extend(
                prob.cpu().numpy()
            )


    val_loss = (
        running_loss
        /
        len(loader.dataset)
    )


    accuracy = accuracy_score(
        all_labels,
        all_preds
    )


    f1 = f1_score(
        all_labels,
        all_preds
    )


    auc = roc_auc_score(
        all_labels,
        all_probs
    )


    eer, eer_threshold = calculate_eer(
        all_labels,
        all_probs
    )


    return {
        "loss": val_loss,
        "accuracy": accuracy,
        "f1": f1,
        "auc": auc,
        "eer": eer,
        "eer_threshold": eer_threshold,
        "labels": np.array(all_labels),
        "preds": np.array(all_preds),
        "probs": np.array(all_probs),
    }

In [ ]:
# AASIST 학습
BEST_MODEL_PATH = (
    "/content/aasist_baseline_best.pt"
)


best_auc = -1


history = []


for epoch in range(
    1,
    EPOCHS + 1
):

    print(
        f"\n"
        f"{'=' * 60}"
    )

    print(
        f"Epoch "
        f"{epoch}/{EPOCHS}"
    )

    print(
        f"{'=' * 60}"
    )


    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        scaler,
        DEVICE,
    )


    val_result = validate(
        model,
        val_loader,
        criterion,
        DEVICE,
    )


    print(
        f"Train Loss : "
        f"{train_loss:.4f}"
    )

    print(
        f"Train Acc  : "
        f"{train_acc:.4f}"
    )

    print(
        f"Valid Loss : "
        f"{val_result['loss']:.4f}"
    )

    print(
        f"Valid Acc  : "
        f"{val_result['accuracy']:.4f}"
    )

    print(
        f"Valid F1   : "
        f"{val_result['f1']:.4f}"
    )

    print(
        f"Valid AUC  : "
        f"{val_result['auc']:.4f}"
    )

    print(
        f"Valid EER  : "
        f"{val_result['eer']:.4f}"
    )


    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_result["loss"],
            "val_acc": val_result["accuracy"],
            "val_f1": val_result["f1"],
            "val_auc": val_result["auc"],
            "val_eer": val_result["eer"],
        }
    )


    if val_result["auc"] > best_auc:

        best_auc = val_result[
            "auc"
        ]

        torch.save(
            {
                "model_state_dict":
                    model.state_dict(),

                "optimizer_state_dict":
                    optimizer.state_dict(),

                "epoch":
                    epoch,

                "auc":
                    best_auc,

                "model_config":
                    model_config,
            },
            BEST_MODEL_PATH,
        )

        print(
            "★ Best model saved"
        )


    gc.collect()

    torch.cuda.empty_cache()

In [ ]:
# 학습결과 확인
history_df = pd.DataFrame(
    history
)

display(
    history_df
)

In [ ]:
# 학습결과 시각화
plt.figure(
    figsize=(8, 5)
)

plt.plot(
    history_df["epoch"],
    history_df["train_loss"],
    label="Train Loss"
)

plt.plot(
    history_df["epoch"],
    history_df["val_loss"],
    label="Valid Loss"
)

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Loss"
)

plt.legend()

plt.show()

In [ ]:
# BEST model
checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE,
)


model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)


print(
    "Best Epoch :",
    checkpoint["epoch"]
)

print(
    "Best AUC   :",
    checkpoint["auc"]
)

In [ ]:
# 최종 validation 성능
final_result = validate(
    model,
    val_loader,
    criterion,
    DEVICE,
)


print("=" * 60)

print("AASIST Baseline Result")

print("=" * 60)

print(
    f"Accuracy : "
    f"{final_result['accuracy']:.4f}"
)

print(
    f"F1       : "
    f"{final_result['f1']:.4f}"
)

print(
    f"ROC-AUC  : "
    f"{final_result['auc']:.4f}"
)

print(
    f"EER      : "
    f"{final_result['eer'] * 100:.2f}%"
)

print(
    f"EER Thr  : "
    f"{final_result['eer_threshold']:.4f}"
)

In [ ]:
cm = confusion_matrix(
    final_result["labels"],
    final_result["preds"]
)


print(
    "Confusion Matrix"
)

print(cm)

In [ ]:
print(
    classification_report(
        final_result["labels"],
        final_result["preds"],
        target_names=[
            "REAL",
            "FAKE"
        ],
        digits=4,
    )
)